In [ ]:
import asyncio
from typing import Optional
from contextlib import AsyncExitStack

from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client
from dotenv import load_dotenv

In [ ]:
server_params = StdioServerParameters(
    command="python",
    args=["server.py"],
    env=None
)
exit_stack = AsyncExitStack()
stdio_transport = await exit_stack.enter_async_context(stdio_client(server_params))
stdio, write = stdio_transport
session = await exit_stack.enter_async_context(ClientSession(stdio,write))
await session.initialize()

In [ ]:
response = await session.list_tools()
available_tools = [{
    "name": tool.name,
    "description": tool.description,
    "input_schema": tool.inputSchema
} for tool in response.tools]

In [ ]:
from llama_cpp import Llama
import json
import requests
from datetime import datetime

# -----------------------------
# Configuration Parameters
# -----------------------------
MODEL_PATH      = "/Users/sayanmandal/workspaces/models/qwen3_1_7b/Qwen3-1.7B-Q8_0.gguf"  # Path to the GGUF model file
N_CTX           = 4096     # Context window size (max number of tokens in context)
N_THREADS       = 8        # Number of CPU threads to use for inference
N_GPU_LAYERS    = 32        # Number of GPU layers to offload (0 = CPU only)
SEED            = 42       # Random seed for reproducibility
USE_MMAP        = True     # Use memory-mapped model file
USE_MLOCK       = False    # Lock model into RAM (requires USE_MMAP=False)
LOW_VRAM        = False    # Reduce VRAM usage (trading compute for memory)
F16_KV          = True     # Use f16 key/value cache
LOGITS_ALL      = False    # Return logits for all tokens
EMBEDDING       = False    # Enable embedding mode
# CHAT_FORMAT     = "chatml-function-calling"

In [ ]:
llm = Llama(
    model_path=MODEL_PATH,
    n_ctx=N_CTX,
    n_threads=N_THREADS,
    n_gpu_layers=N_GPU_LAYERS,
    seed=SEED,
    use_mmap=USE_MMAP,
    use_mlock=USE_MLOCK,
    low_vram=LOW_VRAM,
    f16_kv=F16_KV,
    logits_all=LOGITS_ALL,
    embedding=EMBEDDING,
    # chat_format=CHAT_FORMAT
)

In [ ]:
MAX_TOKENS        = 256      # Maximum tokens to generate
TEMPERATURE       = 0.7      # Sampling temperature
TOP_K             = 50       # Top-K sampling
TOP_P             = 0.95     # Top-P (nucleus) sampling
REPEAT_PENALTY    = 1.1      # Repetition penalty
PRESENCE_PENALTY  = 0.0      # Presence penalty
FREQUENCY_PENALTY = 0.0      # Frequency penalty
STOP_SEQUENCES    = ["\n###"]  # One or more stop sequences
ECHO_INPUT        = False    # Whether to echo the prompt in the output
STREAM            = False    # Streaming output (set to True for callbacks)
LOGPROBS          = None     # Return log probabilities for top tokens

In [ ]:
lines = ""
for t in available_tools:
    lines += f"- {t['name']}: {t['description']}\n\t{{\"tool_name\":\"{t['name']}\", \"parameters\":{t['input_schema']}}}\n"

system_prompt = (
            "A chat between a curious user and an artificial intelligence assistant. The assistant gives helpful, detailed, and polite answers to the user's questions. The assistant calls functions with appropriate input when necessary.\n",
            "You have access to these tools:\n",
            lines,
            "\nWhen you want to call a tool, emit exactly the JSON object by itself, for example:\n",
            "```json\n",
            "{\"tool_call\":{\"tool_name\":\"retrieve_functions\",\"parameters\":{\"query\":\"foo\"}}}\n",
            "```\n",
)
response = llm.create_chat_completion(
      messages = [
        {
          "role": "system",
          "content": ''.join(system_prompt)

        },
        {
          "role": "user",
          "content": "Give me the sum of 107  and 254 and them multiply wirh 5"
        }
      ],
    #   tools=available_tools,
    #   tool_choice='auto'
)

In [ ]:
res_text = response['choices'][0]['message']['content']
# tool_call_info = json.loads(res_text.split('<tool_call>')[-1].removesuffix('</tool_call>').strip())
# tool_name = tool_call_info['name']
# tool_args = tool_call_info['arguments']

In [ ]:
print(res_text)

In [ ]:
print(''.join(system_prompt))

In [ ]:
result = await session.call_tool(tool_name, tool_args)

In [ ]:
available_tools

In [ ]:
print(''.join(system_prompt))

In [ ]:
result.content[0].text

In [ ]:
def greet(name, age):
    print(f"Hello, {name}. You are {age} years old.")

In [ ]:
kwargs = {"name": "Alice", "age": 30}

In [ ]:
greet(**kwargs)